# Custom Model B-v2: Improved Dense Embeddings (Top-5 Sentence Concatenation)

This notebook is an iteration on Model B. Instead of feeding a raw **bag of words** to MiniLM, we extract the **top 5 most informative sentences** per place (scored by TF-IDF) and concatenate them into a single coherent text before encoding.

**Why this is better than mean pooling (v1 attempt):**
- ✅ The Transformer sees complete, sequential, grammatically coherent sentences — exactly the format it was pre-trained on.
- ✅ 5 sentences × ~20 words ≈ 100 words ≈ 130 tokens → safely within MiniLM's 256-token limit.
- ✅ No averaging of 20 disparate vectors (which diluted discriminative features in v1).
- ✅ Single `model.encode()` call: fast and simple.

**Sentence selection strategy:**  
Sentences are scored by the **mean TF-IDF weight** of their words (normalized by sentence length to avoid bias towards long sentences). Only sentences with at least 5 words are considered.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import time
import warnings
warnings.filterwarnings('ignore')

TOP_N_SENTENCES = 5  # Number of best sentences to select and concatenate per place

### 1. Load Raw Data & Train/Test Split (Same Seed as All Other Notebooks)

In [ ]:
# Load raw reviews (NOT prepared_reviews.csv — we need full sentences, not word bags)
df_reviews_raw = pd.read_csv('data/reviews83325.csv')
df_places = pd.read_csv('Tripadvisor.csv', low_memory=False)

# Filter English reviews only
df_reviews_en = df_reviews_raw[df_reviews_raw['langue'] == 'en'].copy()
print(f"English reviews: {len(df_reviews_en)}")

# Group all reviews by place into one big text block
df_grouped = df_reviews_en.groupby('idplace')['review'].apply(
    lambda x: ' '.join(x.dropna().astype(str))
).reset_index()
print(f"Places with English reviews: {len(df_grouped)}")

# Keep only evaluation columns from places metadata
eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places_eval = df_places[eval_cols].copy()

# Merge with place metadata
df_merged = pd.merge(df_grouped, df_places_eval, left_on='idplace', right_on='id', how='inner')
print(f"Places after merge: {len(df_merged)}")

# Deterministic 50/50 train/test split — SAME SEED as all other notebooks
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy().reset_index(drop=True)
test_df  = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Train (queries): {len(train_df)} | Test (search space): {len(test_df)}")

### 2. Evaluation Functions (identical to notebooks 2, 3, and 4)

In [ ]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    cats = []
    cols = ['activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
    for c in cols:
        val = row[c]
        if pd.notna(val):
            parts = str(val).split(',')
            for p in parts:
                cats.append(p.strip().lower())
    return set(cats)

test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]

def eval_level_2(query_subcats, sorted_test_indices):
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        test_sub = test_subcats_list[test_idx]
        if len(query_subcats.intersection(test_sub)) > 0:
            return i
    return None

print("Evaluation functions ready.")

### 3. TF-IDF Sentence Scoring — Build Top-5 Sentence Text per Place

For each place:
1. Split all reviews into individual sentences (split on `.`, `!`, `?`).
2. Fit a `TfidfVectorizer` on those sentences.
3. Score each sentence as the **mean TF-IDF weight of its words** (normalized by sentence length).
4. Keep the top 5 sentences and **concatenate them** into a single text string.

This gives MiniLM 5 coherent, high-information sentences instead of a keyword list.

In [ ]:
def build_top_sentence_text(text, top_n=TOP_N_SENTENCES):
    """Return a single string made of the top_n most informative sentences, scored by TF-IDF."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    # Split into sentences and filter very short ones (< 5 words)
    raw_sentences = [s.strip() for s in text.replace('!', '.').replace('?', '.').split('.')]
    sentences = [s for s in raw_sentences if len(s.split()) >= 5]

    if len(sentences) == 0:
        return text[:512]  # fallback: return truncated raw text

    if len(sentences) <= top_n:
        return '. '.join(sentences)  # fewer sentences than top_n — join all

    try:
        # Fit TF-IDF on all sentences of this place
        vec = TfidfVectorizer(stop_words='english', max_features=500)
        tfidf_matrix = vec.fit_transform(sentences)  # (n_sentences, n_features)

        # Score = mean TF-IDF weight per sentence (naturally normalized per token)
        sentence_scores = np.asarray(tfidf_matrix.mean(axis=1)).ravel()

        # Pick the top_n best, preserve their original order for coherence
        top_indices = sorted(np.argsort(sentence_scores)[::-1][:top_n])
        return '. '.join(sentences[i] for i in top_indices)
    except Exception:
        return '. '.join(sentences[:top_n])  # fallback


print("Building top-sentence texts for all places...")
t0 = time.time()

train_texts = train_df['review'].apply(lambda x: build_top_sentence_text(x, TOP_N_SENTENCES)).tolist()
test_texts  = test_df['review'].apply(lambda x: build_top_sentence_text(x, TOP_N_SENTENCES)).tolist()

print(f"Sentence extraction done in {time.time() - t0:.2f}s")
print(f"\nExample (first train place):")
print(train_texts[0][:300])

### 4. Encode with all-MiniLM-L6-v2

Each place is now represented as a **single coherent text** (≤ ~100 words, safely within the 256-token limit). We use one standard `model.encode()` call — no pooling required.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding Test set...")
t0 = time.time()
test_vectors = model.encode(test_texts, batch_size=64, show_progress_bar=True)
print(f"Test encoding done in {time.time() - t0:.2f}s")

print("\nEncoding Train set (queries)...")
t0 = time.time()
train_vectors = model.encode(train_texts, batch_size=64, show_progress_bar=True)
print(f"Train encoding done in {time.time() - t0:.2f}s")

### 5. Cosine Similarity & Ranking Error Evaluation

In [ ]:
lvl1_errors = []
lvl2_errors = []

start_time = time.time()
print(f"Evaluating {len(train_df)} queries...")

similarities = cosine_similarity(train_vectors, test_vectors)
test_type_array = test_df['typeR'].values

for i in range(len(train_df)):
    row = train_df.iloc[i]
    sims = similarities[i]

    ranked_indices = np.argsort(sims)[::-1]
    test_typeR_list = test_type_array[ranked_indices].tolist()

    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None:
        lvl1_errors.append(err_1)

    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices)
    if err_2 is not None:
        lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 50)
print(f"Model B-v2 (Top-5 Sentence Transformer) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f} (on {len(lvl1_errors)} valid queries)")
print(f"Model B-v2 (Top-5 Sentence Transformer) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f} (on {len(lvl2_errors)} valid queries)")
print("-" * 50)
print("\n--- Comparison Table ---")
print(f"{'Model':<42} {'L1 Error':>10} {'L2 Error':>10}")
print("-" * 64)
print(f"{'BM25 Baseline':<42} {'0.61':>10} {'3.87':>10}")
print(f"{'Model A (TF-IDF + Cosine Sim)':<42} {'0.60':>10} {'4.78':>10}")
print(f"{'Model B (MiniLM on word bag)':<42} {'0.69':>10} {'4.46':>10}")
print(f"{'Model B-v2 (MiniLM + Top-5 Sentences)':<42} {np.mean(lvl1_errors):.2f} {np.mean(lvl2_errors):.2f}")